## Feature Engineering

In [1]:
import pandas as pd
import numpy as np

PATH = "../data/processed/netload/f10_netload_nasa_clean.csv"
df = pd.read_csv(PATH, parse_dates=["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)
df.head()

,Timestamp,MW_raw,NetLoad_MW,ALLSKY_SFC_SW_DWN,T2M,WS10M
0,2025-03-31 15:45:00,1.356,-1.356,425.48,31.07,3.42
1,2025-03-31 16:00:00,0.353,-0.353,194.80,30.73,3.49
2,2025-03-31 16:15:00,-0.509,0.509,194.80,30.73,3.49
3,2025-03-31 16:30:00,-1.902,1.902,194.80,30.73,3.49
4,2025-03-31 16:45:00,-2.148,2.148,194.80,30.73,3.49


### Data Dictionary (Columns & Units)

Timestamp - 15-min date-time

MW_raw - raw feeder MW

NetLoad_MW - net load (MW), = -MW_raw

ALLSKY_SFC_SW_DWN - solar irradiance (W/m²)

T2M - temperature (°C)

WS10M - wind speed (m/s)

In [2]:
print(df.shape)
print(df["Timestamp"].min(), "→", df["Timestamp"].max())
print(df.isna().sum())

(14784, 6)
2025-03-31 15:45:00 → 2025-09-01 15:30:00
Timestamp            0
MW_raw               0
NetLoad_MW           0
ALLSKY_SFC_SW_DWN    0
T2M                  0
WS10M                0
dtype: int64


## 1) Time features (calendar + cyclical)

In [3]:
d = df.copy()

d["hour"] = d["Timestamp"].dt.hour
d["minute"] = d["Timestamp"].dt.minute
d["dow"] = d["Timestamp"].dt.dayofweek      # 0=Mon
d["month"] = d["Timestamp"].dt.month
d["dayofyear"] = d["Timestamp"].dt.dayofyear
d["is_weekend"] = (d["dow"] >= 5).astype(int)

# 15-min slot in day: 0..95
d["slot"] = d["hour"] * 4 + (d["minute"] // 15)

# Cyclical encodings
d["slot_sin"] = np.sin(2*np.pi*d["slot"]/96)
d["slot_cos"] = np.cos(2*np.pi*d["slot"]/96)
d["dow_sin"]  = np.sin(2*np.pi*d["dow"]/7)
d["dow_cos"]  = np.cos(2*np.pi*d["dow"]/7)

## 2) Lag features

In [4]:
target = "NetLoad_MW"
exo = ["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]

lags = [1, 2, 4, 8, 16, 96]

for L in lags:
    d[f"{target}_lag{L}"] = d[target].shift(L)
    for c in exo:
        d[f"{c}_lag{L}"] = d[c].shift(L)

## 3) Rolling statistics (smooth + ramp context)

In [5]:
target = "NetLoad_MW"
windows = [4, 12, 24, 96]

s = d[target].shift(1)  # only past values (prevents leakage)

for w in windows:
    r = s.rolling(window=w, min_periods=w)
    d[f"{target}_roll_mean_{w}"] = r.mean()
    d[f"{target}_roll_std_{w}"]  = r.std(ddof=0)
    d[f"{target}_roll_min_{w}"]  = r.min()
    d[f"{target}_roll_max_{w}"]  = r.max()

In [ ]:
# ramp features
s = d[target].shift(1)          # past value at t-1
d["delta_15m"] = s - s.shift(1) # (t-1)-(t-2)
d["delta_1h"]  = s - s.shift(4) # (t-1)-(t-5)
d["delta_3h"]  = s - s.shift(12)# (t-1)-(t-13)

## 4) Create the day-ahead target (H=96) + drop NaNs

In [7]:
target = "NetLoad_MW"
H = 96  # 96 x 15-min = 1 day ahead

# target = same 15-min slot tomorrow
d["y"] = d[target].shift(-H)

# remove rows with NaNs caused by lags/rolling/target
d_final = d.dropna().reset_index(drop=True)

print("Rows:", len(d_final))
print("Range:", d_final["Timestamp"].min(), "→", d_final["Timestamp"].max())
d_final[["Timestamp", target, "y"]].head()

Rows: 14592
Range: 2025-04-01 15:45:00 → 2025-08-31 15:30:00


,Timestamp,NetLoad_MW,y
0,2025-04-01 15:45:00,-1.460,6.290
1,2025-04-01 16:00:00,-0.843,6.738
2,2025-04-01 16:15:00,0.121,7.203
3,2025-04-01 16:30:00,1.229,7.260
4,2025-04-01 16:45:00,2.193,7.059


## 5) Build X (features) and y (target) without leakage

In [8]:
target = "NetLoad_MW"

# Exclude columns that would leak or are not features
drop_cols = ["Timestamp", "MW_raw", target, "y"]

X_cols = [c for c in d_final.columns if c not in drop_cols]

X = d_final[X_cols]
y = d_final["y"]

print("Num features:", len(X_cols))
print("First 15 feature columns:", X_cols[:15])
print("X shape:", X.shape, "| y shape:", y.shape)

Num features: 57
First 15 feature columns: ['ALLSKY_SFC_SW_DWN', 'T2M', 'WS10M', 'hour', 'minute', 'dow', 'month', 'dayofyear', 'is_weekend', 'slot', 'slot_sin', 'slot_cos', 'dow_sin', 'dow_cos', 'NetLoad_MW_lag1']
X shape: (14592, 57) | y shape: (14592,)


## 6) Time-based train / val / test split (no shuffle)

In [9]:
# ============================================================
# 6) Date-based Train / Validation / Test split (time-series safe)
# ------------------------------------------------------------
# Why date-based split?
# - Time-series forecasting must respect time order (no shuffling).
# - Train uses earliest period (model learns patterns).
# - Validation is a later period (tune hyperparameters / early stopping).
# - Test is the most recent period (final unbiased evaluation).
#
# Split plan:
# - Train:      before 2025-07-15
# - Validation: 2025-07-15 to 2025-07-31
# - Test:       2025-08-01 to end
#
# NOTE:
# - d_final already contains the day-ahead target y = NetLoad_MW(t+96)
# - X and y were already created from d_final with leakage columns removed
# ============================================================

# --- Define split boundaries ---
train_end  = pd.Timestamp("2025-07-15 00:00:00")  # training ends right before this datetime
test_start = pd.Timestamp("2025-08-01 00:00:00")  # test starts from this datetime

# --- Create boolean masks for each split ---
m_train = d_final["Timestamp"] < train_end
m_val   = (d_final["Timestamp"] >= train_end) & (d_final["Timestamp"] < test_start)
m_test  = d_final["Timestamp"] >= test_start

# --- Apply masks to split X (features) and y (target) ---
X_train, y_train = X[m_train], y[m_train]
X_val,   y_val   = X[m_val],   y[m_val]
X_test,  y_test  = X[m_test],  y[m_test]

# --- Print sizes and time ranges to confirm correctness ---
print("Train:", X_train.shape,
      "|", d_final.loc[m_train, "Timestamp"].min(), "→", d_final.loc[m_train, "Timestamp"].max())

print("Val:  ", X_val.shape,
      "|", d_final.loc[m_val, "Timestamp"].min(),   "→", d_final.loc[m_val, "Timestamp"].max())

print("Test: ", X_test.shape,
      "|", d_final.loc[m_test, "Timestamp"].min(),  "→", d_final.loc[m_test, "Timestamp"].max())

Train: (10017, 57) | 2025-04-01 15:45:00 → 2025-07-14 23:45:00
Val:   (1632, 57) | 2025-07-15 00:00:00 → 2025-07-31 23:45:00
Test:  (2943, 57) | 2025-08-01 00:00:00 → 2025-08-31 15:30:00


## Persistence baseline

In [10]:
# ============================================================
# STEP 1: Persistence baseline (Day-ahead, H=96) - leakage safe
# ------------------------------------------------------------
# Task: predict NetLoad_MW at (t + 96 steps) using info at time t
#
# You already created:
#   y(t) = NetLoad_MW(t+96)   via: d_final["y"] = NetLoad_MW.shift(-96)
#
# Persistence baseline rule:
#   y_hat(t) = NetLoad_MW(t)
#
# This is leakage-safe because it uses only the value available at time t
# (no future values are used as inputs).
# ============================================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# --- Use day-ahead target and persistence prediction ---
y_true = d_final["y"].values                # actual NetLoad at (t+96)
y_pred = d_final["NetLoad_MW"].values       # baseline prediction using NetLoad at t

# --- Date-based split boundaries (same as your split plan) ---
train_end  = pd.Timestamp("2025-07-15 00:00:00")
test_start = pd.Timestamp("2025-08-01 00:00:00")

m_train = d_final["Timestamp"] < train_end
m_val   = (d_final["Timestamp"] >= train_end) & (d_final["Timestamp"] < test_start)
m_test  = d_final["Timestamp"] >= test_start

# --- Evaluate on each split ---
for name, mask in [("TRAIN", m_train), ("VAL", m_val), ("TEST", m_test)]:
    yt = y_true[mask.values]
    yp = y_pred[mask.values]
    print(
        f"{name:<5} | MAE: {mean_absolute_error(yt, yp):.4f} "
        f"| RMSE: {rmse(yt, yp):.4f} | N={len(yt)}"
    )

TRAIN | MAE: 1.0205 | RMSE: 1.8616 | N=10017
VAL   | MAE: 0.8641 | RMSE: 1.5147 | N=1632
TEST  | MAE: 1.0264 | RMSE: 1.8320 | N=2943


In [11]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


## Train a LightGBM baseline (day-ahead) using Train/Val split

In [12]:
leaky = [c for c in X_train.columns if ("shift(-" in c) or (c in ["NetLoad_MW","MW_raw","y"])]
print("Potential leakage cols in X:", leaky)

Potential leakage cols in X: []


In [13]:
# ============================================================
# LightGBM Baseline (Day-ahead NetLoad Forecast)
# ------------------------------------------------------------
# Goal:
#   Predict NetLoad one day ahead at 15-min resolution.
#   Your target is: y(t) = NetLoad_MW(t + 96)
#
# Why LightGBM baseline?
# - Strong, standard ML baseline for tabular time-series features
# - Fast to train and easy to compare against your hybrid model
#
# Leakage safety:
# - X contains only past/current-derived features (lags, rolling with shift(1), time features)
# - X does NOT include: NetLoad_MW, MW_raw, or y
# - Train/Val/Test split is time-based (no shuffle)
# ============================================================

import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Helper metric: RMSE (penalizes large errors more than MAE) ---
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# --- Define a reasonable baseline configuration for regression ---
# n_estimators: large number + early stopping chooses best iteration
# learning_rate: small for stable learning
# num_leaves: model complexity (higher can fit more patterns)
# subsample / colsample_bytree: regularization to reduce overfitting
model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# --- Train using Train set, and monitor Validation set for early stopping ---
# early_stopping: stops when val score doesn't improve for 200 rounds
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="l1",  # L1 corresponds to MAE
    callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=True)]
)

# --- Predict on Validation and Test sets ---
pred_val  = model.predict(X_val)
pred_test = model.predict(X_test)

# --- Report metrics ---
print("\nLIGHTGBM BASELINE RESULTS (Day-ahead)")
print("VAL  | MAE:", mean_absolute_error(y_val, pred_val),  "| RMSE:", rmse(y_val, pred_val))
print("TEST | MAE:", mean_absolute_error(y_test, pred_test), "| RMSE:", rmse(y_test, pred_test))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002787 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12115
[LightGBM] [Info] Number of data points in the train set: 10017, number of used features: 57
[LightGBM] [Info] Start training from score 3.989085
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[160]	valid_0's l1: 0.800108	valid_0's l2: 1.64079

LIGHTGBM BASELINE RESULTS (Day-ahead)
VAL  | MAE: 0.8001080825772566 | RMSE: 1.2809321773794056
TEST | MAE: 0.9432898529675668 | RMSE: 1.5237339428806977


In [ ]:
import pandas as pd

results = pd.DataFrame([
    {"Model": "Persistence (ŷ(t)=NetLoad(t))", "Split": "VAL",  "MAE": 0.8641, "RMSE": 1.5147},
    {"Model": "Persistence (ŷ(t)=NetLoad(t))", "Split": "TEST", "MAE": 1.0264, "RMSE": 1.8320},

    {"Model": "LightGBM (57 features)",        "Split": "VAL",  "MAE": 0.8001, "RMSE": 1.2809},
    {"Model": "LightGBM (57 features)",        "Split": "TEST", "MAE": 0.9433, "RMSE": 1.5237},
])

# Show nicely in notebook
results

,Model,Split,MAE,RMSE
0,Persistence (ŷ(t)=NetLoad(t)),VAL,0.8641,1.5147
1,Persistence (ŷ(t)=NetLoad(t)),TEST,1.0264,1.8320
2,LightGBM (57 features),VAL,0.8001,1.2809
3,LightGBM (57 features),TEST,0.9433,1.5237
